# PBSAP Grant Tagging Pipeline

**Organization:** a Philippine conservation foundation (name withheld in this public copy)
**Framework:** 4th Philippine Biodiversity Strategy and Action Plan (PBSAP), 2024–2040 — 5 themes, 23 targets

For each grant, the model returns a shortlist of **three PBSAP targets**, ordered, with a quoted
line of evidence from the grant's own objectives. The notebook attaches each target's official
name, description and theme from a lookup table, checks that the quote really appears in the
objectives, and writes the result to CSV.

> **Public copy.** Outputs are cleared and no grant data is included. The seven worked examples
> in section 2.1 were real tagged grants from the Foundation's portfolio; they are withheld here
> and the list is left empty, so the prompt structure stays visible without publishing partner data.

---

## The notebook runs two files

**Part A — Sample run.**

**Part B — Production run.**

## Order to run

1. Part 0 — Setup
2. Part 1 — PBSAP framework
3. Part 2 — Prompt
4. Part 3 — Pipeline functions
5. **Part A** — sample run, ends with a CSV to review
6. **Part B** — production run, ends with the final CSV

# Part 0 — Setup

### 0.1 Mount Google Drive

Connects the notebook to Drive so it can read the grant files and write output there.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### 0.2 Set the folder and the two input files

Defines where the grant files live and where output is written. `SAMPLE_FILENAME` is the small
file used in Part A; `PRODUCTION_FILENAME` is the full set used in Part B. Both must sit in
`COURSE_FOLDER`, and the names must match Drive exactly, including the `.csv`.

In [ ]:
from pathlib import Path

COURSE_FOLDER = Path("/content/drive/MyDrive/your-folder/csv files")   # set to your own Drive folder

SAMPLE_FILENAME = "raw_untag_2019_2025.csv"                  # Part A: small file, checked by hand
PRODUCTION_FILENAME = "raw_for_tagging_2016_2026.csv"    # Part B: the full untagged set

OUTPUT_FOLDER = COURSE_FOLDER

print("Folder     :", COURSE_FOLDER)
print("Part A file:", SAMPLE_FILENAME)
print("Part B file:", PRODUCTION_FILENAME)

### 0.2b List the folder

Prints what is actually in the folder and flags either filename that is not there, so a typo is
caught now rather than mid-run.

In [ ]:
assert COURSE_FOLDER.exists(), (
    f"Folder not found: {COURSE_FOLDER}\n"
    "Check the spelling in section 0.2, and that Drive finished mounting."
)

print(f"Files in {COURSE_FOLDER}:\n")

for item in sorted(COURSE_FOLDER.iterdir()):
    print(f"  {item.name}")

for label, filename in [("Part A", SAMPLE_FILENAME), ("Part B", PRODUCTION_FILENAME)]:
    if not (COURSE_FOLDER / filename).exists():
        print(f"\nWARNING: {label} file '{filename}' is not in the folder above.")

### 0.3 Install and import the tools

Installs the OpenAI client and imports the libraries the rest of the notebook uses.

In [ ]:
!pip -q install -U openai

import json
import re
import time
from datetime import datetime
from getpass import getpass

import numpy as np
import pandas as pd

from IPython.display import display
from openai import OpenAI

pd.set_option("display.max_colwidth", 150)

print("Imports ready")

### 0.4 Connect to the OpenAI API

Asks for the API key through `getpass`, which hides it while you type and never writes it into
the notebook file, then defines `ask_llm()` — the single function every API call goes through.

Do not paste an API key directly into a cell.

In [ ]:
client = OpenAI(api_key=getpass("OpenAI API key: "))
MODEL = "gpt-4o-mini"


def ask_llm(prompt):
    """Send one prompt to the model and return the text response."""
    try:
        response = client.responses.create(model=MODEL, input=prompt, temperature=0)
    except TypeError:
        response = client.responses.create(model=MODEL, input=prompt)

    return response.output_text


print("OpenAI client ready. Model:", MODEL)

# Part 1 — The PBSAP reference framework

**Do not edit.** These 23 targets and their indicators are what the tagging is measured against.

### 1.1 The framework as data

Defines the 5 themes and 23 targets, each with its name, official description and indicators,
and checks that the numbering is complete. Holding the framework as a table is what lets the
notebook look up a target's name, description and theme from its number, so the model only ever
has to return the number.

In [ ]:
# The 4th PBSAP (2024-2040): 5 themes, 23 targets.
# Source: "PBSAP 4th iteration" reference document.
# Each target keeps its official number. Theme is derived from the target,
# never predicted separately.

PBSAP_THEMES = {
    1: "Theme 1: Ecosystems Conservation",
    2: "Theme 2: Wildlife/Species Conservation",
    3: "Theme 3: Biodiversity Impacts",
    4: "Theme 4: Business and Biodiversity",
    5: "Theme 5: Means of Implementation",
}

PBSAP_TARGETS = {
    1: {
        "theme": 1,
        "name": "Plan and manage all areas to reduce biodiversity loss",
        "statement": (
            "By 2040, at least 24% of terrestrial and inland water, and 16% of "
            "marine and coastal areas accomplished biodiversity inclusive "
            "spatial analysis and planning"
        ),
        "indicators": [
            "Number and area (in ha) of target conservation sites with completed spatial assessment and planning",
            "Number of target sites with approved management plans and detailed spatially-based management strategies",
            "Number and area (in ha) of target sites under effective management",
        ],
    },
    2: {
        "theme": 1,
        "name": "Restore 30% of degraded ecosystems",
        "statement": (
            "By 2040, at least 30% of degraded terrestrial, inland water, coastal, "
            "and marine ecosystems undergo continuous rehabilitation resulting to "
            "enhanced biodiversity, ecosystem functions, ecological integrity, and "
            "connectivity"
        ),
        "indicators": [
            "Number and area (in ha) of degraded and denuded sites rehabilitated and under continuous rehabilitation",
            "Number and area (in ha) of target sites under continuous adoption of threat reduction programs",
        ],
    },
    3: {
        "theme": 1,
        "name": "Conserve 24% of land and 16% of seas",
        "statement": (
            "By 2040, at least 24% of terrestrial and inland water areas, and 16% "
            "of marine and coastal areas are effectively conserved and managed"
        ),
        "indicators": [
            "Number and area (in ha) of target conservation areas under effective management by the designated governance and management bodies",
            "Number of target sites with detailed spatially-based management strategies",
        ],
    },
    12: {
        "theme": 1,
        "name": "Enhance green spaces and urban planning for human well-being and biodiversity",
        "statement": (
            "By 2040, urban green and blue spaces in cities accessible to the "
            "public increased"
        ),
        "indicators": [
            "Number of cities that have completed and adopted the City Biodiversity Index",
            "Number of cities with increased or sustained green and blue spaces",
            "Number of cities compliant with the Seal of Good Local Governance (SGLG) indicators on urban green and blue spaces",
            "Number of policies/guidelines developed for blue and green spaces within the Sustainable Infrastructure Framework",
        ],
    },
    4: {
        "theme": 2,
        "name": "Halt species extinction, protect genetic diversity, and manage human-wildlife conflicts",
        "statement": (
            "By 2040, governance and management regimes are improved for "
            "threatened species, native crops, livestock, wild relatives, and "
            "their genetic resources"
        ),
        "indicators": [
            "Red List Index and national recommendations for inclusion in the list",
            "Number of threatened species that are improving in status",
            "Number of species of priority native crops, livestock, wild relatives, and their genetic resources under effective protection",
            "Number of species conservation plans per flagship/indicator/priority species prepared and implemented",
        ],
    },
    5: {
        "theme": 2,
        "name": "Ensure sustainable, safe and legal harvesting and trade of wild species",
        "statement": (
            "By 2030, the Harvest Control Rules and Reference Points for capture "
            "fisheries under RA 10654 and its IRR are in place"
        ),
        "indicators": [
            "Number of legal wildlife breeders and traders compliant with regulations",
            "Number of economically important species (EIS) in target sites allowed for collection and harvesting",
            "Number of collection permits issued in and outside target sites benefiting resource-dependent local communities",
            "Number of target sites with permanent and effective enforcement programs",
            "Number of airports and seaports with Wildlife and Fishery enforcement desks/units",
        ],
    },
    9: {
        "theme": 2,
        "name": "Manage wild species sustainably to benefit people",
        "statement": (
            "By 2040, wildlife species legally harvested from the wild and bred in "
            "captivity are traded within the allowable limits and conditions set by "
            "RA 9147 (Wildlife Act)"
        ),
        "indicators": [
            "Number of legal wildlife breeders and traders compliant with regulations",
            "Number of economically important species (EIS) in target sites allowed for collection and harvesting",
            "Number of collection permits issued in and outside target sites benefiting resource-dependent local communities",
            "Number of target sites with permanent and effective enforcement programs",
            "Number of airports and seaports with Wildlife and Fishery enforcement desks/units",
        ],
    },
    6: {
        "theme": 2,
        "name": "Reduce the introduction of invasive alien species by 50% and minimize their impacts",
        "statement": (
            "By 2040, the spread of invasive alien species (IAS) reduced by 50% in "
            "protected areas"
        ),
        "indicators": [
            "Number of sites and areas (in ha) where IAS are eradicated/controlled/managed",
            "Number of IAS eradicated, controlled, and reduced in protected areas",
            "Number of IAS operational plans developed and implemented",
        ],
    },
    7: {
        "theme": 3,
        "name": "Reduce pollution to levels that are not harmful to biodiversity",
        "statement": (
            "By 2040, reduce nutrient run-off and pesticide-related impacts on "
            "non-target organisms and ecosystems by at least 50%"
        ),
        "indicators": [
            "Updated and implemented fertilizer and pesticide policy based on agreed industry reform roadmap",
            "Percentage compliance by stakeholders in target sites to standards-based fertilizer and pesticide production and trade management",
            "Volume of importation of allowed synthetic fertilizers and conventional pesticides",
            "Number of LGUs with zero single-use plastic ordinances in target sites",
            "Number of soil quality testing plots in target sites",
            "Number of water quality testing areas in target sites",
            "Number of private companies demonstrating recovery and diversion of plastic packaging",
        ],
    },
    8: {
        "theme": 3,
        "name": "Minimize the impacts of climate change on biodiversity and build resilience",
        "statement": (
            "By 2040, the impacts of climate change in target sites are minimized "
            "and disaster risks reduced through Nature-based Solutions and/or "
            "Ecosystem-based Approaches, and relevant indigenous knowledge systems "
            "and practices"
        ),
        "indicators": [
            "Number of local government units in target sites that adopt NbS and EbA",
            "Number of site governance bodies with management plans including strict climate change and disaster risk protocols",
            "Number of policies issued on NbS and EbA",
        ],
    },
    11: {
        "theme": 3,
        "name": "Restore, maintain and enhance nature's contributions to people",
        "statement": (
            "By 2040, resilience and stability of ecosystems and habitats in target "
            "sites are maintained, especially in vulnerable areas, high exposure "
            "areas, and communities most dependent on biodiversity"
        ),
        "indicators": [
            "Types and sources of ecosystem goods and services (EGS)",
            "Number of target sites under stable and resilient conditions",
            "Number of target sites with Disaster Risk Reduction Management Plans (DRRMP)",
        ],
    },
    17: {
        "theme": 3,
        "name": "Strengthen biosafety and distribute the benefits of biotechnology",
        "statement": "By 2040, biosafety measures and regulations in place and implemented",
        "indicators": [
            "Number of comprehensive biosafety laws and regulations enacted/issued and effectively enforced",
            "Number of biosafety risk assessment tools/systems developed and implemented",
        ],
    },
    10: {
        "theme": 4,
        "name": "Enhance biodiversity and sustainability in agriculture, aquaculture, fisheries, and forestry",
        "statement": (
            "By 2040, agriculture, fisheries, aquaculture, and forestry production "
            "levels in target sites are sustainably managed, and long-term "
            "productivity is maintained"
        ),
        "indicators": [
            "Number of target sites (in ha) managed sustainably and practicing sustainable production in fisheries, aquaculture, agriculture and forestry",
            "Number of target sites (in ha) demonstrating BDFAP and climate-smart agriculture aligned with national and local plans",
            "Number of Nationally Important Agricultural Heritage Systems (NIAHS) sites recognized and sustaining traditional biodiversity-friendly practices",
            "Number of regional and provincial development plans in target sites compliant with NAFMIP guidelines",
        ],
    },
    13: {
        "theme": 4,
        "name": "Increase the sharing of benefits from genetic resources, digital sequence information and traditional knowledge",
        "statement": (
            "By 2040, there is a significant increase of the benefits shared with "
            "providers and rights holders in accordance with applicable ABS "
            "Instruments"
        ),
        "indicators": [
            "Amount or value of monetary and non-monetary benefits received by providers and rights holders, especially IPLCs, women, and youth",
            "Number of permits issued to access genetic resources, associated traditional knowledge, and DSI",
            "Number of local and national policies establishing rights ownership and fair and equitable benefit sharing based on international ABS Instruments",
        ],
    },
    15: {
        "theme": 4,
        "name": "Business assess, disclose and reduce biodiversity-related risks and negative impacts",
        "statement": (
            "By 2040, business and financial entities have adopted an Environmental, "
            "Social, and Governance (ESG) Framework with biodiversity elements and "
            "regularly report and disclose risks, dependencies and impacts on "
            "biodiversity"
        ),
        "indicators": [
            "Number of publicly-listed companies and government financial and non-financial institutions compliant with ESG framework and guidelines",
            "Number of policies on nature-related and agri-based risk disclosures developed and adopted",
            "Number of development projects compliant with established risk disclosure guidelines, biodiversity standards, and safeguards",
        ],
    },
    16: {
        "theme": 4,
        "name": "Enable sustainable consumption choices to reduce waste and overconsumption",
        "statement": (
            "By 2040, policy instruments aimed at supporting the shift to "
            "sustainable consumption and production developed, adopted and "
            "implemented"
        ),
        "indicators": [
            "Number of LGUs, companies, offices, schools, and other sectors practicing circular economy",
            "Number of target sites reporting positive gains in reducing waste and overconsumption",
            "Percentage of food/manufacturing/processing wastes reduced",
        ],
    },
    18: {
        "theme": 4,
        "name": "Eliminate harmful incentives",
        "statement": "By 2040, ecological footprints are reduced/repurposed/redirected or eliminated",
        "indicators": [
            "Number of subsidies and incentives with unintended environmental footprint reduced/repurposed/redirected or eliminated",
            "Number of sector-specific positive incentives introduced to reform programs",
        ],
    },
    14: {
        "theme": 5,
        "name": "Integrate biodiversity in decision-making at every level",
        "statement": (
            "By 2040, biodiversity is fully mainstreamed and integrated in "
            "policies, programs, plans, and processes within all levels of "
            "government and across all sectors"
        ),
        "indicators": [
            "Number of national, regional, and local policies, plans and programs that integrate biodiversity conservation",
            "Number of biodiversity-dependent sectors which adopt biodiversity standards in planning and fiduciary processes",
        ],
    },
    19: {
        "theme": 5,
        "name": "Mobilize financing for biodiversity from all sources, including international finance",
        "statement": (
            "By 2040, financing for biodiversity-related Programs/Activities/Projects "
            "(PAPs) increased to support the annual requirement of the NBSAP"
        ),
        "indicators": [
            "Percentage of the total national budget allocated to biodiversity conservation",
            "Amount of investment in biodiversity conservation reflected in national accounts and local Annual Investment Plans",
            "Percentage of private, ODA, and other sources of funding for biodiversity conservation",
            "Number of innovative funding and fiduciary measures developed and applied",
            "Number of agreements per target site on payment for ecosystem services (PES)",
        ],
    },
    20: {
        "theme": 5,
        "name": "Strengthen capacity-building, technology transfer, and scientific and technical cooperation for biodiversity",
        "statement": (
            "By 2040, mechanisms for continuing capacity-building and technology "
            "transfer are in place"
        ),
        "indicators": [
            "Number of capacity-building and technology transfer programs implemented and maintained across target sectors in target sites",
            "Number of institutions and groups implementing biodiversity-related programs and projects in target sites",
        ],
    },
    21: {
        "theme": 5,
        "name": "Traditional knowledge, awareness, education, and research",
        "statement": (
            "By 2040, interoperability of biodiversity information systems improved, "
            "supported, secured, and accessible to the public"
        ),
        "indicators": [
            "Number of functional knowledge and information platforms such as Clearing House Mechanisms (CHM), Living Planet Index, and IP community-based information systems",
            "Number of users accessing the platforms",
            "Number of security measures on knowledge and information systems in place",
        ],
    },
    22: {
        "theme": 5,
        "name": "Ensure participation in decision-making and access to justice and information related to biodiversity for all",
        "statement": (
            "By 2040, equitable representation and meaningful participation in "
            "decision-making, and access to justice and information related to "
            "biodiversity by gender, youth, Indigenous Peoples, differently abled "
            "persons, local communities, and other stakeholders in target sites"
        ),
        "indicators": [
            "Percentage of IPs and other stakeholders in governing bodies such as PAMB, watershed management councils, forest land management bodies",
        ],
    },
    23: {
        "theme": 5,
        "name": "Ensure gender equality and a gender-responsive approach for biodiversity action",
        "statement": (
            "By 2040, equitable, inclusive, effective, and gender-responsive "
            "representation and participation in decision-making, and access to "
            "justice and information related to biodiversity by all genders, youth, "
            "Indigenous Peoples, differently-abled persons, local communities, and "
            "other stakeholders"
        ),
        "indicators": [
            "Percentage of all genders, youth, Indigenous Peoples, differently-abled persons, local communities, and other stakeholders in biodiversity programs in target sites",
        ],
    },
}

# Structural checks: fail loudly if the framework is edited incorrectly.
assert len(PBSAP_TARGETS) == 23, f"Expected 23 targets, found {len(PBSAP_TARGETS)}"
assert set(PBSAP_TARGETS) == set(range(1, 24)), "Target numbers must be 1-23 with no gaps"

for target_id, target in PBSAP_TARGETS.items():
    assert target["theme"] in PBSAP_THEMES, f"Target {target_id} has an unknown theme"

pbsap_reference = pd.DataFrame(
    [
        {
            "target_id": target_id,
            "target_name": target["name"],
            "theme_id": target["theme"],
            "theme": PBSAP_THEMES[target["theme"]],
            "n_indicators": len(target["indicators"]),
        }
        for target_id, target in sorted(PBSAP_TARGETS.items())
    ]
)

ALLOWED_TARGET_IDS = set(PBSAP_TARGETS)

print("Targets loaded:", len(pbsap_reference))
print("\nTargets per theme:")
display(
    pbsap_reference["theme"]
    .value_counts()
    .sort_index()
    .rename("n_targets")
    .to_frame()
)

display(pbsap_reference)

### 1.2 Render the framework as prompt text

Turns the table above into the block of text injected into every prompt. Building it from the
data means the prompt and the lookup table can never drift apart.

In [ ]:
def build_pbsap_reference_block():
    lines = []

    for theme_id, theme_name in PBSAP_THEMES.items():
        lines.append(f"\n{theme_name.upper()}")

        for target_id, target in sorted(PBSAP_TARGETS.items()):
            if target["theme"] != theme_id:
                continue

            lines.append(f"\nTARGET {target_id:02d} — {target['name']}")
            lines.append(f"  Goal: {target['statement']}")
            lines.append("  Measured by:")

            for indicator in target["indicators"]:
                lines.append(f"    - {indicator}")

    return "\n".join(lines).strip()


PBSAP_REFERENCE_BLOCK = build_pbsap_reference_block()

print(f"Reference block: {len(PBSAP_REFERENCE_BLOCK.split()):,} words\n")
print(PBSAP_REFERENCE_BLOCK[:900])
print("\n... (truncated for display)")

### 1.3 One ambiguity in the source document

Targets 5 and 9 both cover legal and sustainable harvesting and trade of wild species, and the
source document gives them an identical indicator list. Target 5's statement is about capture
fisheries under RA 10654; Target 9 is about wildlife trade under RA 9147. The prompt splits them
on resource type. Grants sitting between the two are ambiguous in the framework itself.

# Part 2 — The classification prompt

### 2.1 Worked examples

In the original run this list held seven real grants with the target a program manager
assigned, chosen to sit on either side of the boundaries the model most often gets wrong
(site management vs. restoration, policy vs. capacity-building, awareness vs. research).
They were injected into every prompt to demonstrate tagging conventions that the PBSAP
document itself does not state.

**In this public copy the seven grants are withheld**, because they are the partner's own
records. Add your own tagged examples in the same `(title, objectives, target)` shape.

In [ ]:
# Real grants with the tag the program managers assigned.
# Chosen to straddle the boundaries the model keeps getting wrong.
#
# Withheld in the public copy. Each entry has this shape:
#   ("<project title>", "<project objectives>", <PBSAP target number>)

WORKED_EXAMPLES = [
    # ("Example title", "Example objectives text ...", 1),
]

WORKED_EXAMPLES_BLOCK = "\n\n".join(
    f"Project title: {title}\n"
    f"Objectives: {objectives}\n"
    f"Assigned by the program manager: TARGET {target:02d}"
    for title, objectives, target in WORKED_EXAMPLES
)

for _, _, target in WORKED_EXAMPLES:
    assert target in PBSAP_TARGETS, f"Invalid target in worked examples: {target}"

print(f"{len(WORKED_EXAMPLES)} worked examples, "
      f"{len(WORKED_EXAMPLES_BLOCK.split()):,} words")

### 2.2 The prompt template

Defines the prompt sent for each grant, in the five-part structure: Task, Context, Input,
Constraints, Output. The constraints carry the budget rule (rank 1 is the target the largest
share of the money goes to) and the tie-break rules for the target pairs the framework confuses.
`build_classification_prompt()` fills in the framework, the worked examples and one grant's
details.

In [ ]:
classification_prompt_template = """
TASK
Rank the three PBSAP targets that best fit the conservation grant below, most likely first.

CONTEXT
The Foundation funds conservation grants in the Philippines and must
report how each grant contributes to the 4th Philippine Biodiversity Strategy and
Action Plan (PBSAP), 2024-2040.

Grants often touch several targets. A program manager assigns just one, so a ranked
shortlist lets a reviewer confirm the right one quickly instead of tagging from scratch.

The full framework is below. Each target belongs to exactly one theme.

{pbsap_reference}

HOW THE FOUNDATION'S PROGRAM MANAGERS HAVE TAGGED SIMILAR GRANTS

{worked_examples}

INPUT
{grant_block}

CONSTRAINTS
- Read the entire project objectives before choosing. Judge the grant by what it sets
  out to achieve overall, not by the first activity mentioned or by a single keyword
  in the text.
- Return exactly three different targets, ranked. Rank 1 is the target that the largest
  share of the grant's budget and activity goes to. Ranks 2 and 3 are the next most
  defensible choices, not filler.
- Do not repeat a target. Do not return fewer than three.
- Grants working on a specific site or protected area default to TARGET 01. This
  includes assessment, spatial planning, management-plan development, site
  management, patrolling and monitoring.
- Choose TARGET 02 only when the budget is spent on physical rehabilitation of
  degraded land or water - nurseries, planting, replanting, assisted natural
  regeneration - not on planning or designing restoration.
- Choose TARGET 03 only when the grant establishes or expands formal protected-area
  coverage, or when the grant is framed around good governance of a conservation area.
- Choose TARGET 14 only when the grant's deliverable is a government policy,
  ordinance, code, law or planning instrument. A grant that merely involves government
  partners, governance structures or coordination is not TARGET 14.
- TARGET 20 covers capacity-building, training, technology transfer, institutional
  strategic planning, and the dissemination of information or raising of awareness.
  TARGET 21 covers studies, research, comparative analysis, conferences and summits,
  traditional knowledge documentation, and knowledge or information platforms. Use the
  deliverable to split them.
- TARGET 22 covers participatory processes and the involvement of Indigenous Peoples,
  youth and other stakeholders in decision-making. If the grant's actual goal is to
  produce a plan rather than to involve stakeholders in one, rank TARGET 01 higher. If
  a youth-focused grant is training rather than participation in decision-making, rank
  TARGET 21 higher.
- TARGET 05 covers capture fisheries harvest control. TARGET 09 covers terrestrial
  wildlife harvesting and trade under the Wildlife Act. Use the resource type.
- TARGET 15 covers private businesses, enterprises and financial institutions
  assessing and disclosing biodiversity risk, including ESG work. Use the actor to
  separate it from TARGET 14.
- These keyword conventions are signals, not overrides. Where a keyword points one way
  and the bulk of the grant's budget and activity points another, follow the budget and
  keep the keyword's target as rank 2.
- Base the answer only on the grant information given. Do not use outside knowledge
  about the grantee.
- Quote evidence directly from the project objectives. Do not paraphrase it.
- Return valid JSON only. No commentary before or after the JSON.

OUTPUT
{{
  "target_ranking": [<first choice>, <second choice>, <third choice>],
  "evidence": "<a direct quote from the project objectives that justifies the first choice>",
  "reasoning": "<one sentence on why rank 1 beats rank 2>"
}}
""".strip()


def build_classification_prompt(grant_block):
    return classification_prompt_template.format(
        pbsap_reference=PBSAP_REFERENCE_BLOCK,
        worked_examples=WORKED_EXAMPLES_BLOCK,
        grant_block=grant_block,
    )


print(f"Prompt scaffold ready ({len(classification_prompt_template.split()):,} words before the framework is inserted)")

# Part 3 — Pipeline functions

Defines every step of the pipeline as a function. Part A and Part B both call these, so the
sample run and the production run are identical.

No API calls are made here — these cells only define functions.

### 3.1 Load and prepare a grant file

`load_grants()` reads a CSV, cleans the header names (spreadsheet exports carry line breaks and
stray spaces inside them), maps the columns onto short working names, adds a `grant_id`, counts
how many words of objectives each grant has, and builds `grant_block` — the only grant
information the model sees.

The amount and attribution-rate columns go into the block. Any existing human tag is read but
never placed in the prompt.

In [ ]:
def clean_header(name):
    """Collapse line breaks and repeated spaces inside a column name."""
    return re.sub(r"\s+", " ", str(name)).strip()


def build_grant_block(row):
    """Assemble the grant description shown to the model."""
    parts = [
        f"Grantee: {row['grantee']}",
        f"Project title: {row['project_title']}",
    ]

    for label, column in [
        ("Location", "location"),
        ("Key Biodiversity Area / Protected Area status", "kba_pa"),
        ("Grant amount requested (PHP)", "amount"),
        ("Biodiversity attribution rate", "attribution_rate"),
    ]:
        value = str(row.get(column, "")).strip()
        if value and value.lower() != "nan":
            parts.append(f"{label}: {value}")

    objectives = str(row.get("project_objectives", "")).strip()
    if objectives.lower() == "nan":
        objectives = ""

    parts.append(f"Project objectives:\n{objectives or '(not provided)'}")

    return "\n".join(parts)


def load_grants(filename):
    """Read one grant CSV and return it ready for tagging."""
    path = COURSE_FOLDER / filename

    assert path.exists(), (
        f"File not found: {path}\nCheck the name against the folder listing in section 0.2b."
    )

    raw = pd.read_csv(path, dtype=str)
    raw.columns = [clean_header(c) for c in raw.columns]

    column_map = {
        "GRANTEE": "grantee",
        "PROJECT TITLE": "project_title",
        "LOCATION": "location",
        "PROJECT OBJECTIVES": "project_objectives",
        "PBSAP Target": "human_pbsap_target",
        "human_tagged_id": "human_target_number",
        "Percentage": "attribution_rate",
    }

    # These two headers vary between exports, so match them by keyword.
    for column in raw.columns:
        if "KBA" in column.upper():
            column_map[column] = "kba_pa"
        elif "AMOUNT" in column.upper() or column.upper() == "TOTAL":
            column_map[column] = "amount"

    frame = raw.rename(columns=column_map).copy()

    required = ["grantee", "project_title", "project_objectives"]
    missing = [c for c in required if c not in frame.columns]

    assert not missing, (
        f"Missing required columns: {missing}\nColumns in the file: {list(raw.columns)}"
    )

    for optional in [
        "location",
        "kba_pa",
        "amount",
        "attribution_rate",
        "human_pbsap_target",
        "human_target_number",
    ]:
        if optional not in frame.columns:
            frame[optional] = np.nan

    frame["grant_id"] = np.arange(1, len(frame) + 1)

    frame["objectives_length"] = (
        frame["project_objectives"].fillna("").astype(str).str.split().str.len()
    )

    frame["grant_block"] = frame.apply(build_grant_block, axis=1)

    return frame


def describe_grants(frame, label):
    """Report row count, blank columns, and how much objectives text there is."""
    print(f"{label}: {len(frame):,} grants\n")

    blanks = {
        column: int(frame[column].fillna("").astype(str).str.strip().eq("").sum())
        for column in ["grantee", "project_title", "location", "kba_pa", "project_objectives"]
    }

    print("Blank cells per column:")
    display(pd.Series(blanks).rename("blank_rows").to_frame())

    print("Words in PROJECT OBJECTIVES:")
    display(frame["objectives_length"].describe().round(1).to_frame("value"))

    thin = frame.loc[frame["objectives_length"] < 15]

    if len(thin) == 0:
        print("Every grant has a usable objectives statement.")
    else:
        print(
            f"{len(thin)} of {len(frame)} grant(s) have fewer than 15 words of objectives. "
            "Those are tagged from the title alone and are flagged for human review."
        )


print("Loading functions defined")

### 3.2 Read the model's reply

`parse_and_validate()` converts one raw model reply into a checked ranking. It strips markdown
code fences, reads the JSON, drops any target number that is not a valid PBSAP target, removes
duplicates, and pads the ranking to three so every row has the same shape. A malformed reply
raises here instead of becoming a bad row in the export.

In [ ]:
def parse_and_validate(raw_text):
    """Turn the model's text response into a checked ranking of three targets.

    Invalid or repeated target numbers are dropped rather than silently kept,
    so a malformed ranking is caught here instead of becoming a bad tag in the
    exported CSV.
    """
    text = str(raw_text).strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()

    start, end = text.find("{"), text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object in response: {text[:200]}")

    result = json.loads(text[start : end + 1])

    ranking_raw = result.get("target_ranking")
    # If target_ranking is not found, check for primary_target_id and create a list
    if ranking_raw is None:
        primary_target_id = result.get("primary_target_id")
        if primary_target_id is not None:
            ranking_raw = [primary_target_id]
        else:
            # If neither target_ranking nor primary_target_id is present, raise error
            raise ValueError(f"Neither 'target_ranking' nor 'primary_target_id' found in response: {text[:200]}")

    if not isinstance(ranking_raw, list):
        raise ValueError(f"target_ranking is not a list: {ranking_raw!r}")

    ranking = []

    for value in ranking_raw:
        if isinstance(value, str) and value.strip().isdigit():
            value = int(value.strip())

        if value in ALLOWED_TARGET_IDS and value not in ranking:
            ranking.append(int(value))

    if not ranking:
        raise ValueError(f"No valid PBSAP targets in the ranking: {ranking_raw!r}")

    # Pad to three so every row has the same shape downstream.
    while len(ranking) < 3:
        ranking.append(None)

    return {
        "ai_target_id": ranking[0],
        "second_choice_number": ranking[1],
        "third_choice_number": ranking[2],
        "n_ranked": sum(1 for r in ranking if r is not None),
        "evidence": str(result.get("evidence", "")).strip(),
        "reasoning": str(result.get("reasoning", "")).strip(),
    }


print("Parser defined")

### 3.3 Tag a set of grants

`tag_grants()` sends one prompt per grant and collects the rankings. A row that fails is retried
once, then recorded with `status="error"` and the message rather than being dropped, so nothing
disappears silently between input and output. Progress is checkpointed to Drive every 25 rows so
a Colab disconnect does not lose a long run.

`limit` caps how many grants are sent — use a small number for a cheap trial.

In [ ]:
PAUSE_SECONDS = 0.0   # raise to 0.5 if you hit rate limits
MAX_ATTEMPTS = 2      # a second try when a call fails or the JSON comes back malformed


def tag_grants(frame, label, limit=None):
    """Send each grant to the model and collect its ranked shortlist."""
    batch = frame if limit is None else frame.head(limit)

    started = datetime.now()
    checkpoint_file = OUTPUT_FOLDER / f"checkpoint_{label}_{started:%Y%m%d_%H%M}.csv"

    records = []

    for position, (_, grant) in enumerate(batch.iterrows(), start=1):
        record = {
            "grant_id": grant["grant_id"],
            "ai_target_id": np.nan,
            "second_choice_number": np.nan,
            "third_choice_number": np.nan,
            "n_ranked": 0,
            "evidence": None,
            "reasoning": None,
            "status": "ok",
            "error": None,
        }

        prompt = build_classification_prompt(grant["grant_block"])

        for attempt in range(1, MAX_ATTEMPTS + 1):
            try:
                record.update(parse_and_validate(ask_llm(prompt)))
                record["status"] = "ok"
                record["error"] = None
                break
            except Exception as exc:
                record["status"] = "error"
                record["error"] = f"{type(exc).__name__}: {exc}"

                if attempt < MAX_ATTEMPTS:
                    time.sleep(1)

        records.append(record)

        if position % 25 == 0 or position == len(batch):
            pd.DataFrame(records).to_csv(checkpoint_file, index=False)
            print(f"  {position} / {len(batch)} done")

        if PAUSE_SECONDS:
            time.sleep(PAUSE_SECONDS)

    results = pd.DataFrame(records)

    print(f"\nFinished in {(datetime.now() - started).total_seconds() / 60:.1f} minutes.")
    print(f"Checkpoint: {checkpoint_file}")

    return results


def report_errors(results):
    """Show which calls failed and which came back with fewer than three targets."""
    display(results["status"].value_counts(dropna=False).rename("count").to_frame())

    incomplete = results.loc[(results["status"] == "ok") & (results["n_ranked"] < 3)]

    if len(incomplete) > 0:
        print(f"{len(incomplete)} row(s) returned fewer than three targets:")
        display(incomplete[["grant_id", "ai_target_id", "n_ranked"]])

    failures = results.loc[results["status"] != "ok"]

    if len(failures) > 0:
        print(f"{len(failures)} row(s) were not tagged:")
        display(failures[["grant_id", "status", "error"]])
    else:
        print("No failed calls.")


print("Tagging functions defined")

### 3.4 Attach the target details

`attach_target_details()` joins each ranked number to its official name, description and theme,
renders the shortlist as one readable cell, and adds two checks:

- `evidence_verified` — whether the model's quote actually appears in the grant's objectives. A
  quote that is not there means the model invented its justification.
- `single_theme` — whether all three choices sit in the same theme. When they do, the model is
  sure of the area and unsure only of the target within it.

`needs_human_review` collects the rows to open first: failed calls, unverified quotes, shortlists
spread across themes, and grants with almost no objectives text.

In [ ]:
def target_field(target_id, field):
    """Look up one field of a PBSAP target. Returns None for missing ranks."""
    if pd.isna(target_id):
        return None

    target = PBSAP_TARGETS[int(target_id)]

    if field == "theme":
        return PBSAP_THEMES[target["theme"]]

    return target[field]


def normalize_for_matching(text):
    """Lowercase and collapse whitespace and punctuation, for quote checking."""
    text = re.sub(r"[^a-z0-9 ]+", " ", str(text).lower())
    return re.sub(r"\s+", " ", text).strip()


def evidence_is_in_objectives(row):
    """Check that the model's quote really appears in the project objectives."""
    quote = str(row.get("evidence") or "").strip()

    if len(quote) < 15:
        return False

    return normalize_for_matching(quote) in normalize_for_matching(row["project_objectives"])


RANK_COLUMNS = ["pbsap_target_number", "second_choice_number", "third_choice_number"]


def ranked_list(row):
    """Render the three choices as one readable cell."""
    parts = []

    for rank, column in enumerate(RANK_COLUMNS, start=1):
        target_id = row[column]

        if pd.isna(target_id):
            continue

        parts.append(f"{rank}. TARGET {int(target_id):02d} - {target_field(target_id, 'name')}")

    return " | ".join(parts)


def attach_target_details(frame, results):
    """Join the rankings back to the grants and add the framework details."""
    # Restrict to the grants that were actually sent, so a limited trial run
    # does not produce a file padded with untagged rows.
    tagged = (
        frame.loc[frame["grant_id"].isin(results["grant_id"])]
        .merge(results, on="grant_id", how="left")
    )

    tagged["pbsap_target_number"] = tagged["ai_target_id"]
    tagged["pbsap_target_name"] = tagged["ai_target_id"].apply(
        lambda t: target_field(t, "name")
    )
    tagged["pbsap_target_description"] = tagged["ai_target_id"].apply(
        lambda t: target_field(t, "statement")
    )
    tagged["pbsap_theme"] = tagged["ai_target_id"].apply(lambda t: target_field(t, "theme"))

    tagged["second_choice_name"] = tagged["second_choice_number"].apply(
        lambda t: target_field(t, "name")
    )
    tagged["third_choice_name"] = tagged["third_choice_number"].apply(
        lambda t: target_field(t, "name")
    )

    tagged["all_three_targets"] = tagged.apply(ranked_list, axis=1)

    tagged["single_theme"] = tagged.apply(
        lambda r: len(
            {target_field(r[c], "theme") for c in RANK_COLUMNS if pd.notna(r[c])}
        )
        == 1,
        axis=1,
    )

    tagged["evidence_verified"] = tagged.apply(evidence_is_in_objectives, axis=1)

    tagged["needs_human_review"] = (
        (tagged["status"] != "ok")
        | ~tagged["evidence_verified"]
        | ~tagged["single_theme"]
        | (tagged["objectives_length"] < 15)
    )

    print(f"Tagged successfully    : {int((tagged['status'] == 'ok').sum())} / {len(tagged)}")
    print(f"Evidence quote found   : {int(tagged['evidence_verified'].sum())} / {len(tagged)}")
    print(f"All three in one theme : {int(tagged['single_theme'].sum())} / {len(tagged)}")
    print(f"Flagged for review     : {int(tagged['needs_human_review'].sum())} / {len(tagged)}")

    return tagged


print("Attach function defined")

### 3.5 Export the tagged CSV

`export_tagged()` selects the output columns, forces the target numbers to whole numbers, and
writes a timestamped CSV into the Drive folder.

In [ ]:
EXPORT_COLUMNS = [
    "grant_id",
    "grantee",
    "project_title",
    "location",
    "kba_pa",
    "project_objectives",
    "amount",
    "pbsap_target_number",
    "pbsap_target_name",
    "pbsap_target_description",
    "pbsap_theme",
    "second_choice_number",
    "second_choice_name",
    "third_choice_number",
    "third_choice_name",
    "all_three_targets",
    "single_theme",
    "evidence",
    "evidence_verified",
    "reasoning",
    "needs_human_review",
    "status",
    "error",
]


def export_tagged(tagged, label):
    """Write the tagged grants to a timestamped CSV in the Drive folder."""
    export = tagged[[c for c in EXPORT_COLUMNS if c in tagged.columns]].copy()

    for column in RANK_COLUMNS:
        if column in export.columns:
            export[column] = export[column].astype("Int64")

    path = OUTPUT_FOLDER / f"{label}_{len(export)}rows_{datetime.now():%Y%m%d_%H%M}.csv"

    export.to_csv(path, index=False)

    print(f"Rows exported : {len(export):,}")
    print(f"Saved to Drive: {path}")

    return export, path


print("Export function defined")

# Part A — Sample run

Tags the small file first and exports it. Open that CSV and check the tags by hand before
starting Part B.

### A.1 Load the sample

Loads `SAMPLE_FILENAME` and reports the row count, any blank columns, and how much objectives
text the model will have to work with.

In [ ]:
sample = load_grants(SAMPLE_FILENAME)

describe_grants(sample, "Sample set")

display(sample[["grant_id", "grantee", "project_title"]].head())

### A.2 Test the prompt on one grant

Sends a single grant and prints the raw reply, then the parsed shortlist. Run this first: a
prompt that fails on one grant fails on all of them, and a mistake found here costs one API call
instead of a whole run.

In [ ]:
TEST_ROW = 0   # change to inspect a different grant

assert 0 <= TEST_ROW < len(sample), f"TEST_ROW must be 0 to {len(sample) - 1}"

test_grant = sample.iloc[TEST_ROW]

print("GRANT")
print("-" * 70)
print(test_grant["grant_block"][:700])

print("\nRAW MODEL RESPONSE")
print("-" * 70)

raw_response = ask_llm(build_classification_prompt(test_grant["grant_block"]))

print(raw_response)

print("\nPARSED")
print("-" * 70)

for key, value in parse_and_validate(raw_response).items():
    print(f"  {key}: {value}")

### A.3 Tag the sample

Sends one API call per grant and collects the shortlists. Set `SAMPLE_LIMIT` to a small number
for an even cheaper first pass, or leave it as `None` to tag the whole sample file.

In [ ]:
SAMPLE_LIMIT = None   # e.g. 10 to tag only the first ten rows; None tags every row

sample_results = tag_grants(sample, "sample", limit=SAMPLE_LIMIT)

### A.4 Check for errors and attach the target details

Reports how many calls succeeded, which rows failed and why, and which came back with fewer than
three targets. Then joins each ranked number to its PBSAP name, description and theme, and
verifies the quoted evidence.

In [ ]:
report_errors(sample_results)

sample_tagged = attach_target_details(sample, sample_results)

display(
    sample_tagged[
        [
            "grant_id",
            "project_title",
            "all_three_targets",
            "evidence_verified",
            "needs_human_review",
        ]
    ].head(15)
)

### A.5 Export the sample — then check it by hand

Writes the sample to a timestamped CSV in the Drive folder.

**Stop here and open that file.** Work through the rows flagged `needs_human_review` first, then
read `evidence` against `project_objectives` on a handful of ordinary rows. If the tags look
wrong in a way that repeats, the fix is a rule in the `CONSTRAINTS` block in Section 2.2 — edit
it, re-run A.3 to A.5, and compare the two exports before going on to Part B.

In [ ]:
sample_export, sample_path = export_tagged(sample_tagged, "sample_tagged_top3")

ok_rows = sample_tagged.loc[sample_tagged["status"] == "ok"]

if len(ok_rows) > 0:
    counts = ok_rows["pbsap_target_number"].astype(int).value_counts().sort_index()

    spread = pd.DataFrame(
        {
            "target_number": counts.index,
            "target_name": [PBSAP_TARGETS[t]["name"] for t in counts.index],
            "n_grants": counts.values,
        }
    ).sort_values("n_grants", ascending=False)

    print("\nFirst choices across the sample:")
    display(spread)

# Part B — Production run

Tags the full untagged set, attaches the target details, and exports the CSV.

Run this only after the sample in Part A has been checked.

### B.1 Load the production file

Loads `PRODUCTION_FILENAME` and reports the same data checks as A.1.

Pay attention to the objectives count. Where objectives are blank the model is tagging from a
project title alone, and those rows carry much less to go on than the sample did.

In [ ]:
production = load_grants(PRODUCTION_FILENAME)

describe_grants(production, "Production set")

display(production[["grant_id", "grantee", "project_title"]].head())

### B.2 Tag the production set

Sends one API call per grant, then reports any failures. Same function as A.3, same prompt, same
model.

In [ ]:
PRODUCTION_LIMIT = None   # e.g. 10 for a trial; None tags every row

production_results = tag_grants(production, "production", limit=PRODUCTION_LIMIT)

print()

report_errors(production_results)

### B.3 Attach the target details

Joins each ranked number to its official PBSAP name, description and theme, verifies the quoted
evidence against the objectives, and flags the rows that need a human to look at them.

In [ ]:
production_tagged = attach_target_details(production, production_results)

display(
    production_tagged[
        [
            "grant_id",
            "grantee",
            "project_title",
            "pbsap_target_number",
            "pbsap_target_name",
            "all_three_targets",
            "needs_human_review",
        ]
    ].head(15)
)

### B.4 Export the tagged CSV

Writes the tagged grants to a timestamped CSV in the Drive folder, then shows how the first
choices are spread across the targets.

In [ ]:
production_export, production_path = export_tagged(production_tagged, "grants_tagged_top3")

ok_rows = production_tagged.loc[production_tagged["status"] == "ok"]

if len(ok_rows) > 0:
    counts = ok_rows["pbsap_target_number"].astype(int).value_counts().sort_index()

    spread = pd.DataFrame(
        {
            "target_number": counts.index,
            "target_name": [PBSAP_TARGETS[t]["name"] for t in counts.index],
            "theme": [PBSAP_THEMES[PBSAP_TARGETS[t]["theme"]] for t in counts.index],
            "n_grants": counts.values,
        }
    ).sort_values("n_grants", ascending=False)

    print("\nFirst choices across the production set:")
    display(spread)

### B.5 Download the file (optional)

Downloads the exported CSV to your computer. The file is already saved in Drive, so this step
can be skipped.

In [ ]:
try:
    from google.colab import files

    files.download(str(production_path))
except Exception as exc:
    print(f"Download skipped ({type(exc).__name__}). The file is saved in Drive.")

# Notes

**What the output is.** A shortlist of three targets per grant, ordered, with a quoted line of
evidence. It is a drafting aid for program managers, not a replacement for their judgment.

**How to read the exported file.** Work through `needs_human_review` first. Then check
`evidence` against `project_objectives` — `evidence_verified` confirms the quote is real, not
that it justifies the target.

**Known limitations.**

1. The human tag reflects which target the largest share of the grant's budget serves. The files
   carry one total amount and no breakdown, so that judgment is only partly available to the
   model.
2. Grants with blank or very short objectives are tagged from a title alone.
3. Targets 5 and 9 share an identical indicator list in the source document, so grants between
   them are ambiguous in the framework itself.
4. One target per grant is enforced at rank 1. Grants genuinely serving several targets are
   flattened, with ranks 2 and 3 only partly compensating.
5. The accuracy figures reported for this project are in-sample: the same tagged grants were
   used to write and check the prompt rules.